In [7]:
import openeo

bbox = {
        "west":  18.51635,
        "south": 48.78376,
        "east":  18.80255,
        "north": 49.04104
        }

In [8]:
connection = openeo.connect(url="openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [10]:
print(connection.describe_collection("SENTINEL2_L2A"))

{'bands': [{'description': 'Coastal aerosol (band 1)', 'eo:common_name': 'coastal', 'gsd': 60, 'name': 'B01'}, {'description': 'Blue (band 2)', 'eo:common_name': 'blue', 'gsd': 10, 'name': 'B02'}, {'description': 'Green (band 3)', 'eo:common_name': 'green', 'gsd': 10, 'name': 'B03'}, {'description': 'Red (band 4)', 'eo:common_name': 'red', 'gsd': 10, 'name': 'B04'}, {'description': 'Red edge 1 (band 5)', 'eo:common_name': 'rededge071', 'gsd': 20, 'name': 'B05'}, {'description': 'Red edge 2 (band 6)', 'eo:common_name': 'rededge075', 'gsd': 20, 'name': 'B06'}, {'description': 'Red edge 3 (band 7)', 'eo:common_name': 'rededge078', 'gsd': 20, 'name': 'B07'}, {'description': 'NIR 1 (band 8)', 'eo:common_name': 'nir', 'gsd': 10, 'name': 'B08'}, {'description': 'NIR 2 (band 8A)', 'eo:common_name': 'nir08', 'gsd': 20, 'name': 'B8A'}, {'description': 'NIR 3 (band 9)', 'eo:common_name': 'nir09', 'gsd': 60, 'name': 'B09'}, {'description': 'SWIR 1 (band 11)', 'eo:common_name': 'swir16', 'gsd': 20,

In [3]:
s2cube = connection.load_collection(
    "SENTINEL2_L2A",
    temporal_extent=["2020-06-01", "2020-07-01"],
    spatial_extent=bbox,
    bands=["B04", "B08", "CLD"],
    #max_cloud_cover= 50
)

In [4]:
ndvis = s2cube.ndvi(nir = "B08", red = "B04", target_band = "NDVI")

In [6]:
cloud_probab = s2cube.band("CLD")
cloud_probab = cloud_probab.resample_cube_spatial(  # lebo CLD ma res 20m ndvi 10m
    ndvis,
    method="near"
)
cloud_mask = cloud_probab < 20 # True → pixel sa zachová ; False → pixel sa zamaskuje (NoData)
ndvis_masked = ndvis.mask(cloud_mask)

In [41]:
#ndvis_masked.merge_cubes(cloud_probab)

In [25]:
#ndvi = s2cube_masked.ndvi(nir = "B08", red = "B04")#, target_band = "NDVI")

In [9]:
# monthly_ndvi = ndvi.aggregate_temporal_period(
#     period="month",
#     reducer="median"
# )

In [43]:
job = ndvis_masked.create_job(
    out_format="NetCDF",
    title="Monthly maskked NDVI with CLD",
    description="NDVI monthly for 2020 (Jun-Oct)"
)
job.start_and_wait()

0:00:00 Job 'j-2604231827494573a90edfea19a735ce': send 'start'
0:00:23 Job 'j-2604231827494573a90edfea19a735ce': created (progress 0%)
0:00:28 Job 'j-2604231827494573a90edfea19a735ce': created (progress 0%)
0:00:35 Job 'j-2604231827494573a90edfea19a735ce': created (progress 0%)
0:00:43 Job 'j-2604231827494573a90edfea19a735ce': created (progress 0%)
0:00:53 Job 'j-2604231827494573a90edfea19a735ce': created (progress 0%)
0:01:05 Job 'j-2604231827494573a90edfea19a735ce': created (progress 0%)
0:01:21 Job 'j-2604231827494573a90edfea19a735ce': created (progress 0%)
0:01:40 Job 'j-2604231827494573a90edfea19a735ce': running (progress N/A)
0:02:04 Job 'j-2604231827494573a90edfea19a735ce': running (progress N/A)
0:02:34 Job 'j-2604231827494573a90edfea19a735ce': running (progress N/A)
0:03:12 Job 'j-2604231827494573a90edfea19a735ce': running (progress N/A)
0:03:59 Job 'j-2604231827494573a90edfea19a735ce': running (progress N/A)
0:04:58 Job 'j-2604231827494573a90edfea19a735ce': running (progress 

<BatchJob job_id='j-2604231827494573a90edfea19a735ce'>

In [44]:
results = job.get_results()
results.download_files("~/ndvi_2020_CLP_asNC")

[PosixPath('~/ndvi_2020_CLP_asNC/openEO.nc'),
 PosixPath('~/ndvi_2020_CLP_asNC/job-results.json')]

In [1]:
job = cloud_probab.create_job(out_format="GTiff",
    title="cloud mask)",
    description="Cloud mask for 2020 (Jun)")
job.start_and_wait()

NameError: name 'cloud_probab' is not defined

In [8]:
results = job.get_results()
results.download_files("~/ndvi_2020_CLD_mask_3")

[PosixPath('~/ndvi_2020_CLD_mask_3/openEO_2020-06-01Z.tif'),
 PosixPath('~/ndvi_2020_CLD_mask_3/openEO_2020-06-06Z.tif'),
 PosixPath('~/ndvi_2020_CLD_mask_3/openEO_2020-06-13Z.tif'),
 PosixPath('~/ndvi_2020_CLD_mask_3/openEO_2020-06-23Z.tif'),
 PosixPath('~/ndvi_2020_CLD_mask_3/job-results.json')]

In [32]:
cloud_probab.download("~/ndvi_2020_CLD_mask_2")

In [9]:
job = cloud_mask.create_job(out_format="GTiff",
    title="cloud mask)",
    description="Cloud mask for 2020 (Jun)")
job.start_and_wait()

0:00:00 Job 'j-2605102206354436bdee825064fdd4f2': send 'start'
0:00:26 Job 'j-2605102206354436bdee825064fdd4f2': created (progress 0%)
0:00:31 Job 'j-2605102206354436bdee825064fdd4f2': created (progress 0%)
0:00:38 Job 'j-2605102206354436bdee825064fdd4f2': created (progress 0%)
0:00:46 Job 'j-2605102206354436bdee825064fdd4f2': queued (progress 0%)
0:00:56 Job 'j-2605102206354436bdee825064fdd4f2': queued (progress 0%)
0:01:08 Job 'j-2605102206354436bdee825064fdd4f2': queued (progress 0%)
0:01:24 Job 'j-2605102206354436bdee825064fdd4f2': queued (progress 0%)
0:01:43 Job 'j-2605102206354436bdee825064fdd4f2': running (progress N/A)
0:02:07 Job 'j-2605102206354436bdee825064fdd4f2': running (progress N/A)
0:02:41 Job 'j-2605102206354436bdee825064fdd4f2': finished (progress 100%)


<BatchJob job_id='j-2605102206354436bdee825064fdd4f2'>

In [10]:
results = job.get_results()
results.download_files("~/ndvi_2020_mask_mask_3")

[PosixPath('~/ndvi_2020_mask_mask_3/openEO_2020-06-01Z.tif'),
 PosixPath('~/ndvi_2020_mask_mask_3/openEO_2020-06-06Z.tif'),
 PosixPath('~/ndvi_2020_mask_mask_3/openEO_2020-06-13Z.tif'),
 PosixPath('~/ndvi_2020_mask_mask_3/openEO_2020-06-23Z.tif'),
 PosixPath('~/ndvi_2020_mask_mask_3/openEO_2020-06-28Z.tif'),
 PosixPath('~/ndvi_2020_mask_mask_3/job-results.json')]